In [1]:
# ============================================================
# Smart Traffic Signal by Dr. Arpit Yadav
# Reactive Agent using LangGraph + Groq + Serper + Gradio
# ============================================================

# =========================
# 1. Install dependencies
# =========================
!pip -q install langgraph langchain langchain-groq gradio requests pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 5.3 MB/s eta 0:00:00


In [2]:

# =========================
# 2. Import libraries
# =========================
import os
import json
import requests
import pandas as pd
from typing import TypedDict, Dict, Any

import gradio as gr

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

In [3]:




# =========================
# 3. API Setup
# =========================
if "GROQ_API_KEY" not in os.environ:
    from getpass import getpass
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ API Key:")

if "SERPER_API_KEY" not in os.environ:
    from getpass import getpass
    os.environ["SERPER_API_KEY"] = getpass("Enter SERPER API Key:")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
SERPER_API_KEY = os.getenv("SERPER_API_KEY")

Enter GROQ API Key:··········
Enter SERPER API Key:··········


In [4]:

# =========================
# 4. Load Groq Model
# =========================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)


In [5]:


# ============================================================
# Reactive Agent Logic
# ============================================================

class TrafficState(TypedDict):
    north: int
    south: int
    east: int
    west: int
    threshold: int
    selected_signal: str
    reasoning: str
    web_info: str
    final_report: str


# =========================
# Sensor Processing Node
# =========================
def sensor_node(state: TrafficState):
    densities = {
        "North": state["north"],
        "South": state["south"],
        "East": state["east"],
        "West": state["west"]
    }
    return {**state, "densities": densities}


# =========================
# Reactive Decision Node
# =========================
def reactive_decision_node(state: TrafficState):
    densities = state["densities"]
    threshold = state["threshold"]

    # PURE REACTIVE RULE
    max_lane = max(densities, key=densities.get)
    max_value = densities[max_lane]

    if max_value >= threshold:
        selected_signal = max_lane
        reason = f"{max_lane} lane has highest density ({max_value}) exceeding threshold"
    else:
        selected_signal = "Round Robin"
        reason = "No lane exceeds threshold → normal cycle"

    return {
        **state,
        "selected_signal": selected_signal,
        "reasoning": reason
    }


# =========================
# Optional Web Search Node
# =========================
def web_search_node(state: TrafficState):
    if not SERPER_API_KEY:
        return {**state, "web_info": "No web info (API missing)"}

    query = "urban traffic congestion control strategies"
    url = "https://google.serper.dev/search"

    payload = json.dumps({"q": query})
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, headers=headers, data=payload)
        data = response.json()
        snippet = data["organic"][0]["snippet"] if "organic" in data else "No data"
    except:
        snippet = "Search failed"

    return {**state, "web_info": snippet}


# =========================
# LLM Explanation Node
# =========================
def llm_node(state: TrafficState):

    prompt = f"""
Explain the reactive traffic signal decision.

Inputs:
North: {state['north']}
South: {state['south']}
East: {state['east']}
West: {state['west']}

Decision: {state['selected_signal']}
Reason: {state['reasoning']}

Explain briefly why this is optimal for real-time traffic control.
"""

    try:
        response = llm.invoke(prompt)
        explanation = response.content
    except:
        explanation = "LLM unavailable"

    return {**state, "llm_explanation": explanation}


# =========================
# Formatter Node
# =========================
def formatter_node(state: TrafficState):

    report = f"""
# 🚦 Smart Traffic Signal Result

## Selected Green Signal
👉 **{state['selected_signal']}**

## Sensor Data
- North: {state['north']}
- South: {state['south']}
- East: {state['east']}
- West: {state['west']}

## Decision Logic
{state['reasoning']}

## AI Explanation
{state.get('llm_explanation','')}

## Traffic Insight
{state.get('web_info','')}

---
⚡ Reactive Agent → Instant decision based on current traffic only
    """

    return {**state, "final_report": report}

In [6]:



# =========================
# 5. LangGraph Workflow
# =========================
builder = StateGraph(TrafficState)

builder.add_node("sensor", sensor_node)
builder.add_node("decision", reactive_decision_node)
builder.add_node("web", web_search_node)
builder.add_node("llm", llm_node)
builder.add_node("format", formatter_node)

builder.set_entry_point("sensor")
builder.add_edge("sensor", "decision")
builder.add_edge("decision", "web")
builder.add_edge("web", "llm")
builder.add_edge("llm", "format")
builder.add_edge("format", END)

traffic_graph = builder.compile()


# =========================
# Run Function
# =========================
def run_traffic(n, s, e, w, threshold):
    state = {
        "north": n,
        "south": s,
        "east": e,
        "west": w,
        "threshold": threshold
    }

    result = traffic_graph.invoke(state)

    return result["final_report"], result["selected_signal"]

In [7]:





# =========================
# 6. Professional Gradio UI
# =========================
css = """
.marquee {
    overflow:hidden;
    white-space:nowrap;
}
.marquee span {
    display:inline-block;
    animation:marquee 10s linear infinite;
    font-weight:bold;
    color:green;
}
@keyframes marquee {
    0% { transform:translateX(100%); }
    100% { transform:translateX(-100%); }
}
"""

with gr.Blocks(css=css) as demo:

    gr.HTML("""
    <div class="marquee">
        <span>Reactive Agent Smart Traffic Signal by Dr. Arpit Yadav 🚦🚦🚦</span>
    </div>
    """)

    gr.Markdown("## 🚦 Smart Traffic Signal (Reactive Agent)")

    with gr.Row():
        north = gr.Slider(0, 100, value=20, label="North Lane Vehicles")
        south = gr.Slider(0, 100, value=40, label="South Lane Vehicles")
        east = gr.Slider(0, 100, value=60, label="East Lane Vehicles")
        west = gr.Slider(0, 100, value=30, label="West Lane Vehicles")

    threshold = gr.Slider(10, 100, value=50, label="Threshold")

    btn = gr.Button("Run Signal Decision")

    output = gr.Markdown()
    signal = gr.Textbox(label="Selected Signal")

    btn.click(
        run_traffic,
        inputs=[north, south, east, west, threshold],
        outputs=[output, signal]
    )

demo.launch(share=True)

/tmp/ipykernel_4162/3575863188.py:21: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=css) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://62743fa31c8f139afc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
